# Setup Prompts Table for BI Hub App

This notebook creates the `prompts` table in Lakebase and grants permissions to the app service principal.

**Prerequisites:**
- Lakebase instance must exist and be available
- App must be deployed (to get service principal ID)

**Run this notebook after:**
1. `databricks bundle deploy`
2. Lakebase instance is created

## Configuration

In [ ]:
# Configuration will be set after pip install and Python restart
# See the next code cell after dependencies are installed

## Install Dependencies

In [ ]:
%pip install "psycopg[binary]" "databricks-sdk>=0.40.0" --upgrade --quiet
dbutils.library.restartPython()

In [ ]:
# Configuration - Update these values as needed
LAKEBASE_INSTANCE = "cx-live-demo-no-delete"
DATABASE_NAME = "databricks_postgres"
APP_NAME = "bi-hub-app"

import uuid
import psycopg
from databricks.sdk import WorkspaceClient

# Initialize Databricks client
wc = WorkspaceClient()
print(f"SDK initialized. Databricks SDK version: {wc.__class__.__module__}")

## Step 1: Verify Lakebase Instance

In [ ]:
# Get Lakebase instance details
# The database API may be under different locations depending on SDK version
try:
    # Try the standard location first
    instance = wc.database.get_database_instance(name=LAKEBASE_INSTANCE)
except AttributeError:
    # Fallback: use the API client directly
    from databricks.sdk.service.database import DatabaseAPI
    db_api = DatabaseAPI(wc.api_client)
    instance = db_api.get_database_instance(name=LAKEBASE_INSTANCE)

print(f"Instance: {instance.name}")
print(f"State: {instance.state}")
print(f"Host: {instance.read_write_dns}")

# Check if instance is available
state_str = str(instance.state)
if "AVAILABLE" not in state_str:
    raise Exception(f"Instance is not available. Current state: {instance.state}")

print("\n✓ Lakebase instance is available")

## Step 2: Get App Service Principal

In [ ]:
# Get app details to find service principal
app = wc.apps.get(APP_NAME)

service_principal_client_id = app.service_principal_client_id
service_principal_name = app.service_principal_name

print(f"App: {app.name}")
print(f"Service Principal Client ID: {service_principal_client_id}")
print(f"Service Principal Name: {service_principal_name}")

if not service_principal_client_id:
    raise Exception("Could not find service principal client ID for app")

print("\n✓ App service principal found")

## Step 3: Generate Database Credentials

In [ ]:
# Generate database credential token
try:
    cred = wc.database.generate_database_credential(
        request_id=str(uuid.uuid4()),
        instance_names=[LAKEBASE_INSTANCE]
    )
except AttributeError:
    from databricks.sdk.service.database import DatabaseAPI
    db_api = DatabaseAPI(wc.api_client)
    cred = db_api.generate_database_credential(
        request_id=str(uuid.uuid4()),
        instance_names=[LAKEBASE_INSTANCE]
    )

# Get current user for connection
current_user = wc.current_user.me().user_name

# Build connection parameters
conn_params = {
    "host": instance.read_write_dns,
    "port": 5432,
    "user": current_user,
    "password": cred.token,
    "dbname": DATABASE_NAME,
    "sslmode": "require"
}

print(f"Connecting as: {current_user}")
print(f"Host: {conn_params['host']}")
print("\n✓ Credentials generated")

## Step 4: Create Prompts Table

In [ ]:
# SQL to create prompts table with seed data
CREATE_PROMPTS_SQL = """
-- Create prompts table
CREATE TABLE IF NOT EXISTS prompts (
    id SERIAL PRIMARY KEY,
    title VARCHAR(255) NOT NULL UNIQUE,
    content TEXT NOT NULL,
    category VARCHAR(50) NOT NULL,
    description TEXT,
    is_favorite BOOLEAN DEFAULT FALSE NOT NULL,
    usage_count INTEGER DEFAULT 0 NOT NULL,
    last_used_at TIMESTAMP,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP NOT NULL,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP NOT NULL,
    CONSTRAINT prompts_category_check CHECK (
        category IN ('customer', 'inventory', 'analytics', 'reporting', 'general')
    )
);

-- Create indexes
CREATE INDEX IF NOT EXISTS ix_prompts_category ON prompts(category);
CREATE INDEX IF NOT EXISTS ix_prompts_is_favorite ON prompts(is_favorite);
CREATE INDEX IF NOT EXISTS ix_prompts_usage_count ON prompts(usage_count);
CREATE INDEX IF NOT EXISTS ix_prompts_last_used_at ON prompts(last_used_at);
CREATE INDEX IF NOT EXISTS ix_prompts_title ON prompts(title);
"""

# Connect and execute
conn = psycopg.connect(**conn_params)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute(CREATE_PROMPTS_SQL)
    print("✓ Prompts table created")

conn.close()

## Step 5: Create Update Trigger

In [ ]:
# SQL for trigger function and trigger
CREATE_TRIGGER_SQL = """
-- Create trigger function for updated_at
CREATE OR REPLACE FUNCTION update_prompts_updated_at()
RETURNS TRIGGER AS $$
BEGIN
    NEW.updated_at = CURRENT_TIMESTAMP;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

-- Create trigger
DROP TRIGGER IF EXISTS prompts_updated_at_trigger ON prompts;
CREATE TRIGGER prompts_updated_at_trigger
    BEFORE UPDATE ON prompts
    FOR EACH ROW
    EXECUTE FUNCTION update_prompts_updated_at();
"""

conn = psycopg.connect(**conn_params)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute(CREATE_TRIGGER_SQL)
    print("✓ Update trigger created")

conn.close()

## Step 6: Seed Initial Prompts

In [ ]:
# Seed data for BI Hub starter prompts
SEED_PROMPTS_SQL = """
INSERT INTO prompts (title, content, category, description, is_favorite) VALUES
-- Customer Behavior (Single-Domain)
('vip_analysis', 'How many VIP customers do we have and what is their average lifetime value compared to other segments?', 'customer', 'Customer Behavior: VIP Analysis', true),
('cart_abandonment', 'What is our cart abandonment rate and how much potential revenue are we leaving on the table?', 'customer', 'Customer Behavior: Cart Abandonment', true),
-- Inventory Operations (Single-Domain)
('inventory_health', 'What is our current inventory health status across all locations, and how much revenue are we losing to stockouts?', 'inventory', 'Inventory Operations: Health Overview', true),
('reorder_priority', 'What products need immediate reordering across our flagship stores? Prioritize by lost sales impact.', 'inventory', 'Inventory Operations: Reorder Priority', true),
-- Voice of Customer (Single-Domain)
('return_patterns', 'What patterns do we see in return feedback? Which issues should we escalate to product teams?', 'customer', 'Voice of Customer: Return Patterns', true),
('brand_love', 'What do customers love most about our brand? What themes emerge from 5-star reviews?', 'customer', 'Voice of Customer: Brand Love', true),
-- Cross-Domain (Multi-Agent Coordination)
('stockout_segments', 'Which of our top-selling product categories have stockout risk, and which customer segments are most affected?', 'analytics', 'Cross-Domain: Stockout Segments', true),
('channel_migration', 'How do our customers migrate between channels, and does our inventory allocation match their channel preferences?', 'analytics', 'Cross-Domain: Channel Migration', true),
('seasonal_trends', 'What product categories are trending with our Loyal customers, and do we have adequate inventory coverage for the upcoming season?', 'analytics', 'Cross-Domain: Seasonal Trends', true),
-- Triple-Agent (Reviews + Behavior + Inventory)
('quality_retention', 'Combine customer sentiment, purchase behavior, and inventory data: What are the top 3 product quality issues affecting VIP customer retention, and do we have inventory coverage for better alternatives?', 'analytics', 'Triple-Agent: Quality & Retention', true)
ON CONFLICT (title) DO NOTHING;
"""

conn = psycopg.connect(**conn_params)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute(SEED_PROMPTS_SQL)
    
    # Check how many prompts exist
    cur.execute("SELECT COUNT(*) FROM prompts")
    count = cur.fetchone()[0]
    print(f"✓ Seed data inserted ({count} prompts total)")

conn.close()

## Step 7: Grant Permissions to App Service Principal

In [ ]:
# Grant permissions to app service principal
# The service principal authenticates using its client_id as the username

GRANT_SQL = f"""
-- Grant schema usage
GRANT USAGE ON SCHEMA public TO "{service_principal_client_id}";

-- Grant table permissions
GRANT SELECT, INSERT, UPDATE, DELETE ON prompts TO "{service_principal_client_id}";

-- Grant sequence permissions (for SERIAL columns)
GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA public TO "{service_principal_client_id}";
"""

conn = psycopg.connect(**conn_params)
conn.autocommit = True

with conn.cursor() as cur:
    for stmt in GRANT_SQL.strip().split(';'):
        stmt = stmt.strip()
        if stmt and not stmt.startswith('--'):
            cur.execute(stmt)
    print(f"✓ Permissions granted to: {service_principal_client_id}")

conn.close()

## Step 8: Verify Setup

In [ ]:
# Verify the setup
conn = psycopg.connect(**conn_params)

with conn.cursor() as cur:
    # Check table exists
    cur.execute("""
        SELECT table_name FROM information_schema.tables 
        WHERE table_schema = 'public' AND table_name = 'prompts'
    """)
    table_exists = cur.fetchone() is not None
    
    # Check row count
    cur.execute("SELECT COUNT(*) FROM prompts")
    prompt_count = cur.fetchone()[0]
    
    # Check grants
    cur.execute(f"""
        SELECT grantee, privilege_type 
        FROM information_schema.table_privileges 
        WHERE table_name = 'prompts' AND grantee = '{service_principal_client_id}'
    """)
    grants = cur.fetchall()

conn.close()

print("=" * 50)
print("SETUP VERIFICATION")
print("=" * 50)
print(f"Table 'prompts' exists: {table_exists}")
print(f"Number of prompts: {prompt_count}")
print(f"Grants for service principal: {len(grants)}")
for grant in grants:
    print(f"  - {grant[1]}")
print("=" * 50)

if table_exists and prompt_count > 0 and len(grants) > 0:
    print("\n✓ Setup complete! The app should now be able to access prompts.")
else:
    print("\n⚠ Setup may be incomplete. Check the values above.")

## Summary

This notebook has:
1. Created the `prompts` table with proper schema
2. Added indexes for performance
3. Created an update trigger for `updated_at`
4. Seeded initial BI Hub prompts
5. Granted permissions to the app service principal

The BI Hub app should now be able to read and write prompts.